## RAG -Tradinitional


### DATA Ingestion - Vector DB PipeLine

In [1]:
from langchain_core.documents import Document

In [2]:
#pdf loader
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_22086/616835706.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
/home/sampath/Documents/ langchain-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_documents(dir_path):
    "load all the pdf files"
    all_pdfs=[]
    path=Path(dir_path)
    pdf_files=list(path.glob("**/*.pdf"))
    for pdf_file in pdf_files:
        print(f"{pdf_file.name} processing")
        try:
            loader=PyPDFLoader(pdf_file)
            pdfs = loader.load()
            for pdf in pdfs:
                pdf.metadata["source"]=pdf_file.name
                pdf.metadata["typr"]="pdf"
            all_pdfs.extend(pdfs)
            print(f"loaded {len(pdfs)}")
        except Exception as e:
            print(f"Error: {e}")
    return all_pdfs 

all_documents=process_documents("/home/sampath/Documents/ langchain-learning/TRAG/data")


Data_visualization_U-2.pdf processing
loaded 26
Data_visualization_U-1.pdf processing
loaded 30
UNIT-4.pdf processing
loaded 16


In [4]:
all_documents

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-02T20:42:27+05:30', 'author': 'Giri Babu Kande', 'moddate': '2026-01-02T20:42:27+05:30', 'source': 'Data_visualization_U-2.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'typr': 'pdf'}, page_content='1 \n \nUNIT-2: Human Perception and Information Processing, What Is Perception , \nPhysiology, Perceptual Processing, Perception in Visualization, Metrics, Cogni-\ntion , Visualization Foundations, The Visualization Process in Detail, Semiology \nof Graphical Symbols, The Eight Visual, Variables Historical Perspective , \nTaxonomies  \n  \n \nHuman Perception and Information Processing \nHuman perception and information processing  play a crucial role in data \nvisualization because visualizations are effective only when they match how \nhumans see, interpret, and understand visual information. The goal of \nvisualization is to reduce cognitive effort and help users q

In [5]:
def split_pdfs(all_documents,chunk_size=2000,chunk_overlap=200):
    "recursively split the documents into chunks"
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_documents=text_splitter.split_documents(all_documents)
    return split_documents

In [6]:
chunks=split_pdfs(all_documents)
chunks

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-02T20:42:27+05:30', 'author': 'Giri Babu Kande', 'moddate': '2026-01-02T20:42:27+05:30', 'source': 'Data_visualization_U-2.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'typr': 'pdf'}, page_content='1 \n \nUNIT-2: Human Perception and Information Processing, What Is Perception , \nPhysiology, Perceptual Processing, Perception in Visualization, Metrics, Cogni-\ntion , Visualization Foundations, The Visualization Process in Detail, Semiology \nof Graphical Symbols, The Eight Visual, Variables Historical Perspective , \nTaxonomies  \n  \n \nHuman Perception and Information Processing \nHuman perception and information processing  play a crucial role in data \nvisualization because visualizations are effective only when they match how \nhumans see, interpret, and understand visual information. The goal of \nvisualization is to reduce cognitive effort and help users q

## Embedding and VectorStoreDb

In [7]:
import numpy as np 
from sentence_transformers import SentenceTransformer
from typing import Dict,Any,List,Tuple
import uuid
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """ Handles document  Embedding generator"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        print(f"Loading embedding model: {self.model_name}")
        self.model=SentenceTransformer(self.model_name)
        print(f"Model loaded Successfully with Embedding Dimention : {self.model.get_embedding_dimension()}")
    def generate_embedding(self,texts:List[str])->np.ndarray:
        print(f"Generaitng Embeddings for {len(texts)} texts")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"embeddings Shape: {embeddings.shape}")
        return embeddings
##Initialize embedding manager
embedding_manager=EmbeddingManager()
print(embedding_manager)


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3351.90it/s]


Model loaded Successfully with Embedding Dimention : 384


In [9]:
import os
from typing import List, Any
import numpy as np
import uuid
import chromadb

class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initializing Chroma DB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            # Creates client which is a reference to the chromadb persistent store
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Creating collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF doc embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error: {e}")
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adding documents and their embeddings to the vector store
        documents: Langchain documents
        embeddings: corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of Documents doesn't match the embeddings")
        print(f"Adding {len(documents)} documents to vector store")

        # Data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Preparing metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i 
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content and embedding list
            documents_text.append(doc.page_content)
            embedding_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error at add_documents function: {e}")
            raise 

vector_store = VectorStore()
vector_store


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 382


In [10]:
# chunks ni embeddings ga change chestunnam

texts=[doc.page_content for doc in chunks]


# generating Embeddings
embeddings=embedding_manager.generate_embedding(texts)

#embeddings lo ki marchaka vector database to store chestam

vector_store.add_documents(chunks,embeddings)

Generaitng Embeddings for 74 texts


Batches: 100%|██████████| 3/3 [00:00<00:00,  3.72it/s]


embeddings Shape: (74, 384)
Adding 74 documents to vector store
Added 74 documents to vector store
Total documents in collection: 456


## Retriever pipeline from VectorStore

In [11]:
class RagRetriever:
    """user gives query and and search it in the vectorstore"""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    def retrieve(self,query:str,top_k:int =5, score_threshold:float =0.0)-> List[Dict[str,Any]]:
        print((f"The Query: {query}"))
        print(f"Top_k= {top_k} and , threshold:{score_threshold}")
        query_embedding=self.embedding_manager.generate_embedding([query])[0]

        #search in vector store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs=[]
            if results["documents"] and results["documents"][0]:
                documents=results["documents"][0]
                metadatas=results["metadatas"][0]
                distances=results["distances"][0]
                ids=results["ids"][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #converting dis to similarity score (chroma Db uses cosine similarity)
                    similarity_score= 1-distance
                    if similarity_score >=score_threshold:
                        retrieved_docs.append({
                            "id":doc_id,
                            "content":document,
                            "metadata":metadata,
                            "similarity_score":similarity_score,
                            "distance": distance,
                            "rank":i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retrieval=RagRetriever(vector_store,embedding_manager)


In [12]:
rag_retrieval.retrieve("Data Types",2)

The Query: Data Types
Top_k= 2 and , threshold:0.0
Generaitng Embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.07it/s]


embeddings Shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_d6300c03_46',
  'content': '20 \n \nClassification Based on Measurement Scale \nData is commonly classified into the following types based on how values are \nmeasured and compared. \n \n1)Nominal Data \nNominal data represents categories or labels with no inherent order . The \nvalues are used only for identification and grouping. \nExamples include gender, blood group, department name, or product \ncategory. \nIn visualization, nominal data is typically represented using: \n• Colour hues \n• Shapes \n• Distinct symbols \nNumerical operations such as addition or averaging are not meaningful for \nnominal data. \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n2) Ordinal Data \nOrdinal data represents categories that have a natural order or ranking, but \nthe differences between values are not measurable. \nExamples include rankings, grades (A, B, C), or satisfaction levels (low, \nmedium, high).',
  'metadata': {'creationdate': '2025-12-13T19:56:17+05:30',


In [13]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
llm=ChatGroq(model="qwen/qwen3-32b", reasoning_format="hidden",temperature=0.11,max_tokens=2024)

In [14]:
def rag_simple(query,retriever,llm,top_k=2):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc["content"] for doc in results]) if results else ""
    prompt= f""" use the following context to answer the question consisely
                content:
                {context}
                Question : {query}
                Answer : 
                along with the score at the bottom 
                """
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content 
    

In [15]:
ans=rag_simple("what is scatter plot",rag_retrieval,llm)
print(ans)

The Query: what is scatter plot
Top_k= 2 and , threshold:0.0
Generaitng Embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 167.95it/s]

embeddings Shape: (1, 384)
Retrieved 2 documents (after filtering)


A scatter plot is a graphical representation used to analyze the relationship between two quantitative variables. It identifies correlations (positive, negative, or none), detects clusters or groupings, highlights outliers, and observes trends/distributions. By encoding data points' positions (x and y axes), it enables precise comparisons. Additional variables can be represented using color (categories), size (magnitude), or shape (classes), allowing multidimensional analysis.  

**Score:** 10/10
